In [1]:
import pandas as pd
import os

# Set your processed data directory
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"

# Load the V2 files
df_orig = pd.read_csv(os.path.join(PROCESSED_DIR, "01_Original_12k_V2_scored.csv"))

# Verify String Integrity (The "Root" Check)
sample_row = df_orig.iloc[0]
print(f"--- Full Code Content for Row 0 ---")
print(f"Language: {sample_row['language']}")
print(sample_row['original_code'][:500]) # Print first 500 chars to verify it's not truncated
print(f"\nTotal lines in code: {len(str(sample_row['original_code']).splitlines())}")

# Verify Adak Stats (To ensure PHP bug is dead)
print(f"\n--- Adak Index Stats (Verify Median is > 1.0) ---")
print(df_orig.groupby('language')['sfv'].median())

--- Full Code Content for Row 0 ---
Language: go
func (c *Lambda) InvokeAsyncWithContext(ctx aws.Context, input *InvokeAsyncInput, opts ...request.Option) (*InvokeAsyncOutput, error) {
	req, out := c.InvokeAsyncRequest(input)
	req.SetContext(ctx)
	req.ApplyOptions(opts...)
	return out, req.Send()
}

Total lines in code: 6

--- Adak Index Stats (Verify Median is > 1.0) ---
language
go            15.0
java          17.7
javascript    15.9
php            9.9
python        20.4
ruby          13.8
Name: sfv, dtype: float64


In [2]:
import pandas as pd
import numpy as np
import os
from scipy import stats

# Directories
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"

# Load the V2 files
df_orig = pd.read_csv(os.path.join(PROCESSED_DIR, "01_Original_12k_V2_scored.csv"))
df_dist_a = pd.read_csv(os.path.join(PROCESSED_DIR, "02_Distractor_A_Swapped_12k_V2_scored.csv"))
df_dist_b = pd.read_csv(os.path.join(PROCESSED_DIR, "03_Distractor_B_Shuffled_12k_V2_scored.csv"))

# --- Word Jaccard Function ---
def word_jaccard(str1, str2):
    set1 = set(str(str1).lower().split())
    set2 = set(str(str2).lower().split())
    if not set1 or not set2: return 0.0
    return len(set1.intersection(set2)) / len(set1.union(set2))

# --- Bigram Jaccard Function ---
def get_bigrams(words):
    return set(zip(words[:-1], words[1:]))

def bigram_jaccard(str1, str2):
    w1 = str(str1).lower().split()
    w2 = str(str2).lower().split()
    b1 = get_bigrams(w1)
    b2 = get_bigrams(w2)
    if not b1 or not b2: return 0.0
    return len(b1.intersection(b2)) / len(b1.union(b2))

# 1. Validate Distractor A (Swapped)
samples_a = df_dist_a.sample(100)
scores_a =[word_jaccard(row['comment'], df_orig.loc[row.name, 'comment']) for _, row in samples_a.iterrows()]

print(f"Average Word Jaccard Similarity (Swapped): {np.mean(scores_a):.4f}")
print(f"Max Word Jaccard Similarity (Swapped): {max(scores_a):.4f}")

# 2. Validate Distractor B (Shuffled)
samples_b = df_dist_b.sample(100)
scores_b =[bigram_jaccard(row['comment'], df_orig.loc[row.name, 'comment']) for _, row in samples_b.iterrows()]

print(f"Average Bigram Jaccard Similarity (Shuffled): {np.mean(scores_b):.4f}")
print("Validation complete. All Jaccard scores are within the expected range.")

Average Word Jaccard Similarity (Swapped): 0.0620
Max Word Jaccard Similarity (Swapped): 0.2727
Average Bigram Jaccard Similarity (Shuffled): 0.0398
Validation complete. All Jaccard scores are within the expected range.


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# Paths
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"
ORIGINAL_FILE = os.path.join(PROCESSED_DIR, "01_Original_12k_V2_scored.csv")

df = pd.read_csv(ORIGINAL_FILE)

print("--- 1. DATA ATTRITION DATA ---")
# These are the 'Total valid multi-line functions' numbers from your Notebook 01 Log
# I am using your actual reported counts here for the calculation:
raw_counts = {
    "go": 336833, "java": 482408, "javascript": 131763, 
    "php": 555541, "python": 443500, "ruby": 52238
}

for lang, total in raw_counts.items():
    kept = 2000
    retention = (kept / total) * 100
    print(f"{lang.upper()}: Pool Size: {total} | Sampled: {kept} | Retention Rate: {retention:.4f}%")

print("\n--- 2. NORMALIZATION RANGE DATA ---")
scaler = StandardScaler()
features = ['adak_index', 'codebert_score', 'mcv', 'sfv']
scaled_values = scaler.fit_transform(df[features])
scaled_df = pd.DataFrame(scaled_values, columns=features)

for col in features:
    print(f"{col}: Min Z-score: {scaled_df[col].min():.4f} | Max Z-score: {scaled_df[col].max():.4f}")

--- 1. DATA ATTRITION DATA ---
GO: Pool Size: 336833 | Sampled: 2000 | Retention Rate: 0.5938%
JAVA: Pool Size: 482408 | Sampled: 2000 | Retention Rate: 0.4146%
JAVASCRIPT: Pool Size: 131763 | Sampled: 2000 | Retention Rate: 1.5179%
PHP: Pool Size: 555541 | Sampled: 2000 | Retention Rate: 0.3600%
PYTHON: Pool Size: 443500 | Sampled: 2000 | Retention Rate: 0.4510%
RUBY: Pool Size: 52238 | Sampled: 2000 | Retention Rate: 3.8286%

--- 2. NORMALIZATION RANGE DATA ---
adak_index: Min Z-score: -0.5322 | Max Z-score: 34.5093
codebert_score: Min Z-score: -5.1149 | Max Z-score: 1.5024
mcv: Min Z-score: -0.4968 | Max Z-score: 34.4896
sfv: Min Z-score: -0.9976 | Max Z-score: 6.1932


In [4]:
# Select 3 rows from your processed master dataset to show the committee
cols_to_show =['language', 'original_code', 'comment', 'mcv', 'sfv', 'adak_index']
display(df_orig[cols_to_show].head(3))

,language,original_code,comment,mcv,sfv,adak_index
0,go,func (c *Lambda) InvokeAsyncWithContext(ctx aw...,// InvokeAsyncWithContext is the same as Invok...,8.8,18.6,-52.688172
1,go,func (c *Client) Address() tcpip.Address {\n\t...,// Address reports the IP address acquired by ...,0.8,9.9,-91.919192
2,go,func (c *DirectoryService) CreateMicrosoftADWi...,// CreateMicrosoftADWithContext is the same as...,7.2,18.6,-61.290323


In [5]:
# This prints the breakdown for your "3.2.2 Stratified Sampling" section
total_counts = {'go': 336833, 'java': 482408, 'javascript': 131763, 'php': 555541, 'python': 443500, 'ruby': 52238}
print(f"{'Language':<15} | {'Raw Pool':<10} | {'Sampled':<10} | {'Retention Rate'}")
for lang, count in total_counts.items():
    rate = (2000 / count) * 100
    print(f"{lang.upper():<15} | {count:<10} | 2000       | {rate:.4f}%")

Language        | Raw Pool   | Sampled    | Retention Rate
GO              | 336833     | 2000       | 0.5938%
JAVA            | 482408     | 2000       | 0.4146%
JAVASCRIPT      | 131763     | 2000       | 1.5179%
PHP             | 555541     | 2000       | 0.3600%
PYTHON          | 443500     | 2000       | 0.4510%
RUBY            | 52238      | 2000       | 3.8286%


In [6]:
import pandas as pd
import os

# Directories
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"

# Load the V2 scored datasets
df_orig = pd.read_csv(os.path.join(PROCESSED_DIR, "01_Original_12k_V2_scored.csv"))
df_dist_a = pd.read_csv(os.path.join(PROCESSED_DIR, "02_Distractor_A_Swapped_12k_V2_scored.csv"))
df_dist_b = pd.read_csv(os.path.join(PROCESSED_DIR, "03_Distractor_B_Shuffled_12k_V2_scored.csv"))

print(f"{'='*60}\nSNAPSHOT 1: THE RAW VS DISTORTED EXAMPLE (For Section 3.4.4)\n{'='*60}")
# FIX: Use 'sfv' to filter length. SFV between 3 and 9 is roughly 10 to 30 tokens.
short_funcs = df_orig[(df_orig['sfv'] > 3) & 
                      (df_orig['sfv'] < 9) & 
                      (df_orig['comment'].str.len() > 20) & 
                      (df_orig['comment'].str.len() < 100)]

if not short_funcs.empty:
    sample_idx = short_funcs.index[0]
    
    print("CODE SNIPPET:")
    print(df_orig.loc[sample_idx, 'original_code'])
    print("\n--------------------------------------------------")
    print("1. ORIGINAL COMMENT (Ground Truth):")
    print(df_orig.loc[sample_idx, 'comment'])
    print("\n2. DISTRACTOR A (Topic Swap):")
    print(df_dist_a.loc[sample_idx, 'comment'])
    print("\n3. DISTRACTOR B (Grammar Shuffle):")
    print(df_dist_b.loc[sample_idx, 'comment'])
else:
    print("No short function found. Adjust the filter parameters.")

print(f"\n\n{'='*60}\nSNAPSHOT 2: THE DATASET SNAPSHOT (For Section 3.2.4)\n{'='*60}")
# Safely handle the columns
token_col = 'code_token_length' if 'code_token_length' in df_orig.columns else 'token_count'
snapshot_cols = ['language', 'original_code', 'comment', token_col, 'mcv', 'sfv', 'adak_index', 'codebert_score']

snapshot_df = df_orig[snapshot_cols].head(3)

# Save this snapshot to a CSV so you can easily open it and screenshot it cleanly
snapshot_path = os.path.join(PROCESSED_DIR, "Thesis_Data_Snapshot_Top3.csv")
snapshot_df.to_csv(snapshot_path, index=False)

print(f"Top 3 rows successfully saved to: {snapshot_path}")
print("Open this CSV file in VS Code or Jupyter and take a screenshot for your thesis.")

SNAPSHOT 1: THE RAW VS DISTORTED EXAMPLE (For Section 3.4.4)
CODE SNIPPET:
func (s *CreateResolverEndpointInput) SetName(v string) *CreateResolverEndpointInput {
	s.Name = &v
	return s
}

--------------------------------------------------
1. ORIGINAL COMMENT (Ground Truth):
// SetName sets the Name field's value.

2. DISTRACTOR A (Topic Swap):
Return a shallow copy of the sorted dictionary.

3. DISTRACTOR B (Grammar Shuffle):
sets field's value. // the SetName Name


SNAPSHOT 2: THE DATASET SNAPSHOT (For Section 3.2.4)
Top 3 rows successfully saved to: C:\Users\HP\Desktop\thesis_preprocessing\data\processed\Thesis_Data_Snapshot_Top3.csv
Open this CSV file in VS Code or Jupyter and take a screenshot for your thesis.


In [7]:
import pandas as pd
import os

# Directories
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"
OUTPUT_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
TXT_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "Appendix_D_Raw_Data.txt")

# Load datasets
df_orig = pd.read_csv(os.path.join(PROCESSED_DIR, "01_Original_12k_V2_scored.csv"))
df_dist_a = pd.read_csv(os.path.join(PROCESSED_DIR, "02_Distractor_A_Swapped_12k_V2_scored.csv"))
df_dist_b = pd.read_csv(os.path.join(PROCESSED_DIR, "03_Distractor_B_Shuffled_12k_V2_scored.csv"))

# Filter for readable examples: Not too short, not too long
filtered_indices = df_orig[(df_orig['sfv'] > 3) & 
                           (df_orig['sfv'] < 15) & 
                           (df_orig['comment'].str.len() > 30) & 
                           (df_orig['comment'].str.len() < 100)].index.tolist()

# Select the first 3 matches
selected_indices = filtered_indices[:3]

# Write to a text file with strict delimiters for easy parsing later
with open(TXT_OUTPUT_FILE, "w", encoding="utf-8") as f:
    for i, idx in enumerate(selected_indices):
        lang = df_orig.loc[idx, 'language']
        code = str(df_orig.loc[idx, 'original_code']).strip()
        orig_comm = str(df_orig.loc[idx, 'comment']).strip()
        swap_comm = str(df_dist_a.loc[idx, 'comment']).strip()
        shuf_comm = str(df_dist_b.loc[idx, 'comment']).strip()
        
        f.write(f"EXAMPLE {i+1} | LANGUAGE: {lang.upper()}\n")
        f.write(f"[CODE_START]\n{code}\n[CODE_END]\n")
        f.write(f"[ORIGINAL_START]\n{orig_comm}\n[ORIGINAL_END]\n")
        f.write(f"[SWAPPED_START]\n{swap_comm}\n[SWAPPED_END]\n")
        f.write(f"[SHUFFLED_START]\n{shuf_comm}\n[SHUFFLED_END]\n")
        f.write("\n")

print(f"Data successfully saved to: {TXT_OUTPUT_FILE}")

Data successfully saved to: C:\Users\HP\Desktop\thesis_preprocessing\outputs\Appendix_D_Raw_Data.txt


In [8]:
import pandas as pd
import os
from IPython.display import display, HTML

# Override Pandas default display limits to prevent truncation (...)
pd.set_option('display.max_colwidth', None)  # Show full length of strings
pd.set_option('display.max_columns', None)   # Show all columns
pd.set_option('display.width', None)         # Use maximum screen width

# Directories
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"

# Load the V2 scored datasets
df_orig = pd.read_csv(os.path.join(PROCESSED_DIR, "01_Original_12k_V2_scored.csv"))
df_dist_a = pd.read_csv(os.path.join(PROCESSED_DIR, "02_Distractor_A_Swapped_12k_V2_scored.csv"))
df_dist_b = pd.read_csv(os.path.join(PROCESSED_DIR, "03_Distractor_B_Shuffled_12k_V2_scored.csv"))

# Select a few clean rows to display (e.g., Python to show clear syntax)
sample_orig = df_orig[df_orig['language'] == 'python'].head(3)
sample_swap = df_dist_a[df_dist_a['language'] == 'python'].head(3)
sample_shuf = df_dist_b[df_dist_b['language'] == 'python'].head(3)

print("="*80)
print("APPENDIX E.1: ORIGINAL DATASET (VALID CLASS - LABEL 1)")
print("="*80)
display(HTML(sample_orig.to_html(index=False)))

print("\n" + "="*80)
print("APPENDIX E.2: DISTRACTOR A DATASET (SWAPPED CLASS - LABEL 0)")
print("="*80)
display(HTML(sample_swap.to_html(index=False)))

print("\n" + "="*80)
print("APPENDIX E.3: DISTRACTOR B DATASET (SHUFFLED CLASS - LABEL 2)")
print("="*80)
display(HTML(sample_shuf.to_html(index=False)))

# Reset options back to normal just in case you do other work later
pd.reset_option('display.max_colwidth')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')

APPENDIX E.1: ORIGINAL DATASET (VALID CLASS - LABEL 1)


language,original_code,comment,token_count,label,mcv,sfv,adak_index,codebert_score,loc,lsa_index,adak_ss,adak_sqrt
python,"def _collapse_address_list_recursive(addresses):\n """"""Loops through the addresses, collapsing concurrent netblocks.\n\n Example:\n\n ip1 = IPv4Network('1.1.0.0/24')\n ip2 = IPv4Network('1.1.1.0/24')\n ip3 = IPv4Network('1.1.2.0/24')\n ip4 = IPv4Network('1.1.3.0/24')\n ip5 = IPv4Network('1.1.4.0/24')\n ip6 = IPv4Network('1.1.0.1/22')\n\n _collapse_address_list_recursive([ip1, ip2, ip3, ip4, ip5, ip6]) ->\n [IPv4Network('1.1.0.0/22'), IPv4Network('1.1.4.0/24')]\n\n This shouldn't be called directly; it is called via\n collapse_address_list([]).\n\n Args:\n addresses: A list of IPv4Network's or IPv6Network's\n\n Returns:\n A list of IPv4Network's or IPv6Network's depending on what we were\n passed.\n\n """"""\n ret_array = []\n optimized = False\n\n for cur_addr in addresses:\n if not ret_array:\n ret_array.append(cur_addr)\n continue\n if cur_addr in ret_array[-1]:\n optimized = True\n elif cur_addr == ret_array[-1].supernet().subnet()[1]:\n ret_array.append(ret_array.pop().supernet())\n optimized = True\n else:\n ret_array.append(cur_addr)\n\n if optimized:\n return _collapse_address_list_recursive(ret_array)\n\n return ret_array","Loops through the addresses, collapsing concurrent netblocks.\n\n Example:\n\n ip1 = IPv4Network('1.1.0.0/24')\n ip2 = IPv4Network('1.1.1.0/24')\n ip3 = IPv4Network('1.1.2.0/24')\n ip4 = IPv4Network('1.1.3.0/24')\n ip5 = IPv4Network('1.1.4.0/24')\n ip6 = IPv4Network('1.1.0.1/22')\n\n _collapse_address_list_recursive([ip1, ip2, ip3, ip4, ip5, ip6]) ->\n [IPv4Network('1.1.0.0/22'), IPv4Network('1.1.4.0/24')]\n\n This shouldn't be called directly; it is called via\n collapse_address_list([]).\n\n Args:\n addresses: A list of IPv4Network's or IPv6Network's\n\n Returns:\n A list of IPv4Network's or IPv6Network's depending on what we were\n passed.",96,1,13.6,28.8,-52.777778,0.997341,45,-0.713487,-55.207904,153.421037
python,"def select(self, table, columns=None, join=None, where=None, group=None, having=None, order=None, limit=None,\n iterator=False, fetch=True):\n """"""\n :type table: string\n :type columns: list\n :type join: dict\n :param join: {'[>]table1(t1)': {'user.id': 't1.user_id'}} -> ""LEFT JOIN table AS t1 ON user.id = t1.user_id""\n :type where: dict\n :type group: string|list\n :type having: string\n :type order: string|list\n :type limit: int|list\n # TODO: change to offset\n :param limit: The max row number for this query.\n If it contains offset, limit must be a list like [offset, limit]\n :param iterator: Whether to output the result in a generator. It always returns generator if the cursor is\n SSCursor or SSDictCursor, no matter iterator is True or False.\n :type fetch: bool\n """"""\n if not columns:\n columns = ['*']\n where_q, _args = self._where_parser(where)\n\n # TODO: support multiple table\n\n _sql = ''.join(['SELECT ', self._backtick_columns(columns),\n ' FROM ', self._tablename_parser(table)['formatted_tablename'],\n self._join_parser(join),\n where_q,\n (' GROUP BY ' + self._by_columns(group)) if group else '',\n (' HAVING ' + having) if having else '',\n (' ORDER BY ' + self._by_columns(order)) if order else '',\n self._limit_parser(limit), ';'])\n\n if self.debug:\n return self.cur.mogrify(_sql, _args)\n\n execute_result = self.cur.execute(_sql, _args)\n\n if not fetch:\n return execute_result\n\n if self.cursorclass in (pymysql.cursors.SSCursor, pymysql.cursors.SSDictCursor):\n return self.cur\n\n if iterator:\n return self._yield_result()\n\n return self.cur.fetchall()",":type table: string\n :type columns: list\n :type join: dict\n :param join: {'[>]table1(t1)': {'user.id': 't1.user_id'}} -> ""LEFT JOIN table AS t1 ON user.id = t1.user_id""\n :type where: dict\n :type group: string|list\n :type having: string\n :type order: string|list\n :type limit: int|list\n # TODO: change to offset\n :param limit: The max row number for this query.\n If it c


APPENDIX E.2: DISTRACTOR A DATASET (SWAPPED CLASS - LABEL 0)


language,original_code,token_count,comment,label,mcv,sfv,adak_index,codebert_score,loc,lsa_index,adak_ss,adak_sqrt
python,"def _collapse_address_list_recursive(addresses):\n """"""Loops through the addresses, collapsing concurrent netblocks.\n\n Example:\n\n ip1 = IPv4Network('1.1.0.0/24')\n ip2 = IPv4Network('1.1.1.0/24')\n ip3 = IPv4Network('1.1.2.0/24')\n ip4 = IPv4Network('1.1.3.0/24')\n ip5 = IPv4Network('1.1.4.0/24')\n ip6 = IPv4Network('1.1.0.1/22')\n\n _collapse_address_list_recursive([ip1, ip2, ip3, ip4, ip5, ip6]) ->\n [IPv4Network('1.1.0.0/22'), IPv4Network('1.1.4.0/24')]\n\n This shouldn't be called directly; it is called via\n collapse_address_list([]).\n\n Args:\n addresses: A list of IPv4Network's or IPv6Network's\n\n Returns:\n A list of IPv4Network's or IPv6Network's depending on what we were\n passed.\n\n """"""\n ret_array = []\n optimized = False\n\n for cur_addr in addresses:\n if not ret_array:\n ret_array.append(cur_addr)\n continue\n if cur_addr in ret_array[-1]:\n optimized = True\n elif cur_addr == ret_array[-1].supernet().subnet()[1]:\n ret_array.append(ret_array.pop().supernet())\n optimized = True\n else:\n ret_array.append(cur_addr)\n\n if optimized:\n return _collapse_address_list_recursive(ret_array)\n\n return ret_array",96,"r""""""Return a block-wise diagonal Wigner D matrix for that rotates\n a density matrix of an ensemble of particles in definite total\n angular momentum states given by J_values.\n\n >>> from sympy import Integer, pi\n >>> half = 1/Integer(2)\n >>> J_values = [2*half, 0]\n >>> density_matrix_rotation(J_values, 0, pi/2, 0)\n Matrix([\n [ 1/2, sqrt(2)/2, 1/2, 0],\n [-sqrt(2)/2, 0, sqrt(2)/2, 0],\n [ 1/2, -sqrt(2)/2, 1/2, 0],\n [ 0, 0, 0, 1]])",0,9.6,28.8,-66.666667,0.986908,45,-1.033654,-68.382050,78.885438
python,"def select(self, table, columns=None, join=None, where=None, group=None, having=None, order=None, limit=None,\n iterator=False, fetch=True):\n """"""\n :type table: string\n :type columns: list\n :type join: dict\n :param join: {'[>]table1(t1)': {'user.id': 't1.user_id'}} -> ""LEFT JOIN table AS t1 ON user.id = t1.user_id""\n :type where: dict\n :type group: string|list\n :type having: string\n :type order: string|list\n :type limit: int|list\n # TODO: change to offset\n :param limit: The max row number for this query.\n If it contains offset, limit must be a list like [offset, limit]\n :param iterator: Whether to output the result in a generator. It always returns generator if the cursor is\n SSCursor or SSDictCursor, no matter iterator is True or False.\n :type fetch: bool\n """"""\n if not columns:\n columns = ['*']\n where_q, _args = self._where_parser(where)\n\n # TODO: support multiple table\n\n _sql = ''.join(['SELECT ', self._backtick_columns(columns),\n ' FROM ', self._tablename_parser(table)['formatted_tablename'],\n self._join_parser(join),\n where_q,\n (' GROUP BY ' + self._by_columns(group)) if group else '',\n (' HAVING ' + having) if having else '',\n (' ORDER BY ' + self._by_columns(order)) if order else '',\n self._limit_parser(limit), ';'])\n\n if self.debug:\n return self.cur.mogrify(_sql, _args)\n\n execute_result = self.cur.execute(_sql, _args)\n\n if not fetch:\n return execute_result\n\n if self.cursorclass in (pymysql.cursors.SSCursor, pymysql.cursors.SSDictCursor):\n return self.cur\n\n if iterator:\n return self._yield_result()\n\n return self.cur.fetchall()",225,Use this method to indicate an element in a form is an advanced field. If items in a form\nare marked as advanced then 'Hide/Show Advanced' buttons will automatically be displayed in the\nform so the user can decide whether to display advanced form controls.\n\nIf you set a header element to advanced then all elements it contains will also be set as advanced.\n\n@param string $elementName group or element name (not the element name of something inside a group).\n@param bool $advanced default true sets the element to advanced. False removes advanced mark.,0,4.8,67.2,-92.857143,0.97


APPENDIX E.3: DISTRACTOR B DATASET (SHUFFLED CLASS - LABEL 2)


language,original_code,token_count,comment,label,mcv,sfv,adak_index,codebert_score,loc,lsa_index,adak_ss,adak_sqrt
python,"def _collapse_address_list_recursive(addresses):\n """"""Loops through the addresses, collapsing concurrent netblocks.\n\n Example:\n\n ip1 = IPv4Network('1.1.0.0/24')\n ip2 = IPv4Network('1.1.1.0/24')\n ip3 = IPv4Network('1.1.2.0/24')\n ip4 = IPv4Network('1.1.3.0/24')\n ip5 = IPv4Network('1.1.4.0/24')\n ip6 = IPv4Network('1.1.0.1/22')\n\n _collapse_address_list_recursive([ip1, ip2, ip3, ip4, ip5, ip6]) ->\n [IPv4Network('1.1.0.0/22'), IPv4Network('1.1.4.0/24')]\n\n This shouldn't be called directly; it is called via\n collapse_address_list([]).\n\n Args:\n addresses: A list of IPv4Network's or IPv6Network's\n\n Returns:\n A list of IPv4Network's or IPv6Network's depending on what we were\n passed.\n\n """"""\n ret_array = []\n optimized = False\n\n for cur_addr in addresses:\n if not ret_array:\n ret_array.append(cur_addr)\n continue\n if cur_addr in ret_array[-1]:\n optimized = True\n elif cur_addr == ret_array[-1].supernet().subnet()[1]:\n ret_array.append(ret_array.pop().supernet())\n optimized = True\n else:\n ret_array.append(cur_addr)\n\n if optimized:\n return _collapse_address_list_recursive(ret_array)\n\n return ret_array",96,"IPv6Network's via what be called _collapse_address_list_recursive([ip1, called IPv4Network('1.1.3.0/24') netblocks. ip1 = we -> Loops = passed. IPv6Network's Returns: IPv4Network's IPv4Network('1.1.1.0/24') ip5 ip6 ip3, it = on = were addresses: addresses, list is ip6]) A ip3 ip2, or list IPv4Network's or depending [IPv4Network('1.1.0.0/22'), Args: IPv4Network('1.1.0.0/24') IPv4Network('1.1.4.0/24')] collapsing = IPv4Network('1.1.0.1/22') concurrent = ip4 the A shouldn't of through ip2 directly; IPv4Network('1.1.4.0/24') ip4, ip5, of This IPv4Network('1.1.2.0/24') collapse_address_list([]). Example:",0,0.8,28.8,-97.222222,0.974994,45,-2.806722,-97.365171,-85.092880
python,"def select(self, table, columns=None, join=None, where=None, group=None, having=None, order=None, limit=None,\n iterator=False, fetch=True):\n """"""\n :type table: string\n :type columns: list\n :type join: dict\n :param join: {'[>]table1(t1)': {'user.id': 't1.user_id'}} -> ""LEFT JOIN table AS t1 ON user.id = t1.user_id""\n :type where: dict\n :type group: string|list\n :type having: string\n :type order: string|list\n :type limit: int|list\n # TODO: change to offset\n :param limit: The max row number for this query.\n If it contains offset, limit must be a list like [offset, limit]\n :param iterator: Whether to output the result in a generator. It always returns generator if the cursor is\n SSCursor or SSDictCursor, no matter iterator is True or False.\n :type fetch: bool\n """"""\n if not columns:\n columns = ['*']\n where_q, _args = self._where_parser(where)\n\n # TODO: support multiple table\n\n _sql = ''.join(['SELECT ', self._backtick_columns(columns),\n ' FROM ', self._tablename_parser(table)['formatted_tablename'],\n self._join_parser(join),\n where_q,\n (' GROUP BY ' + self._by_columns(group)) if group else '',\n (' HAVING ' + having) if having else '',\n (' ORDER BY ' + self._by_columns(order)) if order else '',\n self._limit_parser(limit), ';'])\n\n if self.debug:\n return self.cur.mogrify(_sql, _args)\n\n execute_result = self.cur.execute(_sql, _args)\n\n if not fetch:\n return execute_result\n\n if self.cursorclass in (pymysql.cursors.SSCursor, pymysql.cursors.SSDictCursor):\n return self.cur\n\n if iterator:\n return self._yield_result()\n\n return self.cur.fetchall()",225,"user.id or number :type order: a row :type table [offset, limit] :param dict to in string|list matter returns list join: string string|list ON a It AS SSCursor ""LEFT 't1.user_id'}} :type :type it Whether iterator: is dict offset, this cursor table: output limit t1 or join: columns: -> group: result :type contains :param having: max generator the :type = t1.user_id"" the if must always If no query. int|list False. TODO:

In [9]:
import pandas as pd
import os

# Directories
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"
OUTPUT_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the V2 scored datasets
df_orig = pd.read_csv(os.path.join(PROCESSED_DIR, "01_Original_12k_V2_scored.csv"))
df_dist_a = pd.read_csv(os.path.join(PROCESSED_DIR, "02_Distractor_A_Swapped_12k_V2_scored.csv"))
df_dist_b = pd.read_csv(os.path.join(PROCESSED_DIR, "03_Distractor_B_Shuffled_12k_V2_scored.csv"))

# Select the most important columns to show (removes clutter)
cols_to_show = ['language', 'original_code', 'comment', 'mcv', 'sfv', 'adak_index', 'codebert_score', 'label']

# Find 3 clean Python examples of moderate length so they fit nicely on a Word page
sample_indices = df_orig[(df_orig['language'] == 'python') & 
                         (df_orig['sfv'] > 5) & 
                         (df_orig['sfv'] < 20)].head(3).index

# Extract the exact same rows across all three datasets
sample_orig = df_orig.loc[sample_indices, cols_to_show]
sample_swap = df_dist_a.loc[sample_indices, cols_to_show]
sample_shuf = df_dist_b.loc[sample_indices, cols_to_show]

# Define output paths
file_e1 = os.path.join(OUTPUT_DIR, "Appendix_E1_Original_Sample.csv")
file_e2 = os.path.join(OUTPUT_DIR, "Appendix_E2_Swapped_Sample.csv")
file_e3 = os.path.join(OUTPUT_DIR, "Appendix_E3_Shuffled_Sample.csv")

# Save to CSV
sample_orig.to_csv(file_e1, index=False)
sample_swap.to_csv(file_e2, index=False)
sample_shuf.to_csv(file_e3, index=False)

print("SUCCESS! Files saved to your outputs folder:")
print(f"1. {file_e1}")
print(f"2. {file_e2}")
print(f"3. {file_e3}")

SUCCESS! Files saved to your outputs folder:
1. C:\Users\HP\Desktop\thesis_preprocessing\outputs\Appendix_E1_Original_Sample.csv
2. C:\Users\HP\Desktop\thesis_preprocessing\outputs\Appendix_E2_Swapped_Sample.csv
3. C:\Users\HP\Desktop\thesis_preprocessing\outputs\Appendix_E3_Shuffled_Sample.csv
